# 🎙️➡️🎨 From Voice to Vision — Fase 5: Pipeline potenziata + Ri-ottimizzazione

La Fase 4 ha rivelato un **overfitting del validation set** (val alta ma test in calo).
Qui lo correggiamo e ri-ottimizziamo:
- **Validation a 4 attori** (meno rumoroso) + fitness = media delle 3 migliori epoche
- **SpecAugment** (mascheramento tempo/frequenza) sul training
- **Canali delta + delta-delta** (dinamica della prosodia) → input a 3 canali
- **Scheduler del learning rate** (ReduceLROnPlateau)

Obiettivo: il modello ottimizzato deve **battere il baseline** (0.53) e recuperare le classi difficili.

In [ ]:
# === 1. Repo aggiornata + feature + NUOVO split ===
REPO_URL = "https://github.com/<tuo-username>/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git","")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm

import numpy as np
from src import config, data_loader, features
data_loader.download_ravdess()
df = data_loader.build_index()
data = features.build_dataset(df, denoise=False, cache=True)   # feature dalla cache

# le feature non dipendono dallo split: applichiamo il nuovo split (4 attori in val)
assert np.array_equal(data["y"], df["emotion_id"].to_numpy()), "ordine feature/df non allineato"
data["split"] = df["split"].to_numpy()
splits = features.split_arrays(data)
print("Nuovo split (speaker-independent):")
for s in ["train","val","test"]:
    print(f"  {s:5s}: {len(splits[s]['y'])} clip | attori {sorted(df[df.split==s].actor.unique())}")

## 1) Verifica veloce: pipeline potenziata con iperparametri di default

In [ ]:
from src.models import cnn
print("Device:", cnn.get_device())
hp_default = {"lr":1e-3, "dropout":0.3, "weight_decay":1e-4, "batch_size":32, "width":32}
check = cnn.train_cnn(splits, hp_default, epochs=45, patience=12,
                      deltas=True, augment=True, verbose=True)
print(f"\nDefault + pipeline potenziata → test = {check['test_acc']:.3f}  (baseline era 0.529)")

## 2) Ri-ottimizzazione con la pipeline potenziata
Stessi tre algoritmi, ma ora la fitness è robusta e il training usa augmentation + delta.
⏱️ ~12-18 min ciascuno su T4. Metti `QUICK=True` per una prova rapida.

In [ ]:
QUICK = False
if QUICK:
    N_AGENTS, N_ITER, EPOCHS_SEARCH = 5, 3, 8
else:
    N_AGENTS, N_ITER, EPOCHS_SEARCH = 6, 6, 14

from src.optimization import pso, fso, ga
from src.optimization.search_space import make_objective, decode, DIM
import time

def run(optimizer, **kw):
    obj = make_objective(splits, epochs=EPOCHS_SEARCH, patience=5)
    t0 = time.time()
    res = optimizer.optimize(obj, DIM, seed=config.SEED, **kw)
    res["time_s"] = time.time()-t0
    res["n_evals"] = len(obj.history_evals)
    res["best_hp"] = decode(res["best_u"])
    print(f"→ {res['name']}: robust_val={res['best_fit']:.3f} | eval={res['n_evals']} | {res['time_s']:.0f}s")
    return res

### PSO

In [ ]:
res_pso = run(pso, n_particles=N_AGENTS, n_iter=N_ITER); print(res_pso['best_hp'])

### FSO (Flock of Starlings)

In [ ]:
res_fso = run(fso, n_particles=N_AGENTS, n_iter=N_ITER); print(res_fso['best_hp'])

### GA

In [ ]:
res_ga = run(ga, pop_size=N_AGENTS, n_iter=N_ITER); print(res_ga['best_hp'])

## 3) Confronto convergenza + modello finale

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
allres = [res_pso, res_fso, res_ga]
plt.figure(figsize=(9,5))
for r in allres: plt.plot(range(len(r['history'])), r['history'], marker='o', label=r['name'])
plt.xlabel('iterazione'); plt.ylabel('robust validation accuracy')
plt.title('Convergenza (pipeline potenziata): PSO vs FSO vs GA'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR/'05_convergence.png', dpi=120); plt.show()

tab = pd.DataFrame([{ 'algoritmo':r['name'],'robust_val':round(r['best_fit'],3),
                      'valutazioni':r['n_evals'],'tempo_s':round(r['time_s']), **r['best_hp']} for r in allres])
display(tab)

In [ ]:
# Riaddestramento completo della configurazione migliore
import json, torch, seaborn as sns
best = max(allres, key=lambda r: r['best_fit'])
print(f"Vince: {best['name']} (robust_val={best['best_fit']:.3f}) | hp={best['best_hp']}")
final = cnn.train_cnn(splits, best['best_hp'], epochs=70, patience=15,
                      deltas=True, augment=True, verbose=True)
print(f"\n*** CNN ottimizzata+potenziata — TEST = {final['test_acc']:.3f} ***")
print(final['report'])

plt.figure(figsize=(7,6))
sns.heatmap(final['cm'], annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=config.EMOTIONS, yticklabels=config.EMOTIONS)
plt.title(f"CNN finale ({best['name']}) — test={final['test_acc']:.2f}")
plt.xlabel('predetto'); plt.ylabel('reale'); plt.xticks(rotation=45)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR/'05_confusion_final.png', dpi=120); plt.show()

## 4) Miglioramento rispetto al baseline

In [ ]:
improv = pd.DataFrame([
    {'modello':'SVM (MFCC)','test_acc':0.517},
    {'modello':'CNN default','test_acc':0.529},
    {'modello':'CNN default + pipeline pot.','test_acc':round(check['test_acc'],3)},
    {'modello':f"CNN ottimizzata ({best['name']})",'test_acc':round(final['test_acc'],3)},
])
plt.figure(figsize=(9,4))
sns.barplot(data=improv, x='test_acc', y='modello', color='#4C78A8')
for i,v in enumerate(improv['test_acc']): plt.text(v+0.01, i, f"{v:.2f}", va='center')
plt.xlim(0,1); plt.title('Progressione dell\'accuracy sul test (speaker-independent)')
plt.tight_layout(); plt.savefig(config.FIGURES_DIR/'05_improvement.png', dpi=120); plt.show()
display(improv)

# salvataggi per Fase 6 (XAI) e report
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR/'best_hp.json','w') as f:
    json.dump({'winner':best['name'],'hp':best['best_hp'],
               'robust_val':best['best_fit'],'test_acc':final['test_acc']}, f, indent=2)
with open(config.RESULTS_DIR/'optimization_results.json','w') as f:
    json.dump([{ 'name':r['name'],'robust_val':r['best_fit'],'n_evals':r['n_evals'],
                 'time_s':r['time_s'],'history':r['history'],'hp':r['best_hp']} for r in allres], f, indent=2)
torch.save({'state_dict':final['model'].state_dict(),'hp':best['best_hp'],
            'norm':final['norm'],'in_channels':final['in_channels']}, config.RESULTS_DIR/'best_cnn.pt')
print('✓ salvati best_hp.json, optimization_results.json, best_cnn.pt')

### ✅ Fase 5 completata
Ora l'ottimizzazione dovrebbe battere il baseline in modo netto e le classi difficili migliorano.

**Mandami:** la tabella di confronto, la curva di convergenza, l'accuracy finale e la barra del miglioramento.
Poi in **Fase 6** apriamo la scatola nera con **Grad-CAM, t-SNE e SHAP** (explainability).